# 03 Exploratory Data Analysis

This notebook creates default-rate summary tables and simple bar charts for key LendingClub loan fields.

## 1. Import Libraries

In [ ]:
# Import libraries for data analysis and visualization
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Load Clean Data

In [ ]:
# Set input and output paths
data_path = "../data/processed/loans_clean_v1.csv"
tables_dir = Path("../reports/tables")
figures_dir = Path("../reports/figures")

# Create output folders if they do not already exist
tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

# Load the clean modeling dataset
loans = pd.read_csv(data_path, low_memory=False)

print(f"Rows: {loans.shape[0]:,}")
print(f"Columns: {loans.shape[1]:,}")

### Milestone 1 Analysis: EDA Dataset

**Observation**  
The EDA notebook uses the cleaned version 1 dataset rather than the raw LendingClub file.

**Interpretation**  
This means the target definition and basic cleaning decisions are already applied before risk-driver analysis.

**Modeling Implication**  
EDA should be performed on the same population that will feed the first PD model so that observed risk patterns match the modeling sample.

**Decision**  
Use `loans_clean_v1.csv` for default-rate summaries and future model development.

## 3. Create Summary and Chart Helper

In [ ]:
def create_default_rate_summary(data, group_column):
    """Create a default-rate summary table for one column."""
    summary = (
        data.groupby(group_column, dropna=False)
        .agg(
            number_of_loans=("default_flag", "size"),
            number_of_defaults=("default_flag", "sum"),
            default_rate=("default_flag", "mean")
        )
        .reset_index()
    )

    # Sort by default rate so higher-risk groups are easier to see
    summary = summary.sort_values(by="default_rate", ascending=False)

    return summary


def save_default_rate_outputs(data, group_column):
    """Save one summary table and one bar chart for a grouping column."""
    summary = create_default_rate_summary(data, group_column)

    # Save the summary table
    table_path = tables_dir / f"default_rate_by_{group_column}.csv"
    summary.to_csv(table_path, index=False)

    # Create a simple bar chart of default rate by group
    plt.figure(figsize=(10, 5))
    plt.bar(summary[group_column].astype(str), summary["default_rate"], color="steelblue")
    plt.title(f"Default Rate by {group_column}")
    plt.xlabel(group_column)
    plt.ylabel("Default Rate")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    # Save the chart
    figure_path = figures_dir / f"default_rate_by_{group_column}.png"
    plt.savefig(figure_path, dpi=150)
    plt.show()

    print(f"Saved table: {table_path}")
    print(f"Saved figure: {figure_path}")

    return summary

### Milestone 1 Analysis: Summary Method

**Observation**  
Each EDA table calculates loan count, default count, and default rate for one grouped variable.

**Interpretation**  
This format shows both risk level and sample size. A high default rate based on very few loans should be interpreted cautiously.

**Modeling Implication**  
Useful PD features should show meaningful risk separation while also having enough observations to support stable model learning.

**Decision**  
Use these summaries to identify candidate risk drivers, not as final proof that a variable belongs in the model.

## 4. Default Rate by Grade

In [ ]:
grade_summary = save_default_rate_outputs(loans, "grade")
grade_summary

### Milestone 1 Analysis: Grade Risk Driver

**Observation**  
The grade summary compares default rates across LendingClub credit grades.

**Interpretation**  
Credit grade is expected to be strongly related to borrower risk because it already reflects LendingClub's assessment of credit quality.

**Modeling Implication**  
Grade may be predictive, but it should be reviewed carefully because it can summarize information from other borrower attributes and may dominate simpler models.

**Decision**  
Treat `grade` as an important candidate feature, while checking whether the project goal allows using LendingClub-assigned grades.

## 5. Default Rate by Sub Grade

In [ ]:
sub_grade_summary = save_default_rate_outputs(loans, "sub_grade")
sub_grade_summary

### Milestone 1 Analysis: Sub-Grade Risk Driver

**Observation**  
The sub-grade summary gives a more granular view of risk than grade.

**Interpretation**  
Sub-grades may reveal finer risk separation within each broad grade.

**Modeling Implication**  
`sub_grade` can improve predictive power, but it may be highly correlated with `grade` and interest rate.

**Decision**  
Consider `sub_grade` as a candidate feature, but avoid blindly using several overlapping LendingClub risk labels without checking redundancy.

## 6. Default Rate by Term

In [ ]:
term_summary = save_default_rate_outputs(loans, "term")
term_summary

### Milestone 1 Analysis: Term Risk Driver

**Observation**  
The term summary compares default rates across loan durations.

**Interpretation**  
Longer loans may carry different risk because borrowers have more time to experience financial stress and because longer terms can be associated with different borrower profiles.

**Modeling Implication**  
`term` is a strong candidate feature for the first PD model because it is known at origination and has a direct relationship to loan structure.

**Decision**  
Keep `term` in the candidate feature set.

## 7. Default Rate by Purpose

In [ ]:
purpose_summary = save_default_rate_outputs(loans, "purpose")
purpose_summary

### Milestone 1 Analysis: Purpose Risk Driver

**Observation**  
The purpose summary compares default rates across loan purposes.

**Interpretation**  
Loan purpose may capture differences in borrower need, financial pressure, and use of funds.

**Modeling Implication**  
`purpose` is a categorical risk-driver candidate, but rare categories may need grouping before modeling.

**Decision**  
Keep `purpose` for EDA and consider category consolidation during feature engineering.

## 8. Default Rate by Home Ownership

In [ ]:
home_ownership_summary = save_default_rate_outputs(loans, "home_ownership")
home_ownership_summary

### Milestone 1 Analysis: Home Ownership Risk Driver

**Observation**  
The home ownership summary compares default rates across borrower housing categories.

**Interpretation**  
Housing status can be related to borrower stability, financial obligations, and credit profile.

**Modeling Implication**  
`home_ownership` is available at origination and may be useful, but categories with small counts should be handled carefully.

**Decision**  
Keep `home_ownership` as a candidate categorical feature for the first PD model.